# 10. The hydrogen spectrum and the Rydberg constant

**Domain:** atomic physics / spectroscopy.

Hydrogen emission wavelengths obey

$$\frac{1}{\lambda} = R_H\left(\frac{1}{n_1^2} - \frac{1}{n_2^2}\right)$$

For the **Balmer series** ($n_1 = 2$, the visible lines):

$$\frac{1}{\lambda} = \frac{R_H}{4} - R_H\cdot\frac{1}{n^2}$$

linear in $1/n^2$, with slope $-R_H$ and intercept $R_H/4$. Both carry the same
constant, giving a built-in consistency check.

**Goal:** recover $1/n^2$ and measure $R_H$ — twice, independently.

### The two-stage rule

`beamfeat` fits **ridge** regression (`alpha=1.0`) on *standardised* features, which
shrinks a single coefficient by about $n/(n+\alpha)$. Fine for prediction, fatal for
measuring a constant. So:

1. **`beamfeat` finds the form.**
2. **OLS on that form estimates the constant.**

Never read a physical constant off `.equation()`.


## Setup


In [ ]:
%pip install -q "beamfeat[units]" pandas matplotlib seaborn scikit-learn scipy


In [ ]:
import warnings
from scipy import constants as C
from sklearn.linear_model import LinearRegression
from beamfeat import BeamFeatRegressor

warnings.filterwarnings("ignore", message=".*valid feature names.*")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CSV_DIR = Path("csv")
CSV_DIR.mkdir(exist_ok=True)          # created on first run, reused after


def save_csv(frame, name):
    # Write to csv/<name> once, then read back so every run starts from disk.
    path = CSV_DIR / name
    if not path.exists():
        frame.to_csv(path, index=False)
        print(f"wrote  {path}  ({len(frame)} rows)")
    else:
        print(f"cached {path}")
    return pd.read_csv(path)


def raw_coef(m):
    # beamfeat standardises internally; rescale to original units.
    return m.coef_ / m.scaler_.scale_


def pct_err(est, true):
    return 100.0 * (est - true) / true

sns.set_theme(style="whitegrid")


## Get the constant right: reduced mass

`scipy.constants.Rydberg` is $R_\infty$ — the value for an *infinitely heavy* nucleus.
Real hydrogen has a finite-mass proton, so the electron orbits a shared centre of mass
and the correct constant is

$$R_H = \frac{R_\infty}{1 + m_e/m_p}$$

The correction is only 0.055%, but that is larger than our measurement precision, so
ignoring it would be a genuine error.


In [ ]:
R_inf = C.Rydberg
R_H = R_inf / (1 + C.m_e / C.m_p)

print(f"R_inf (scipy) : {R_inf:,.3f} 1/m")
print(f"R_H  (proton) : {R_H:,.3f} 1/m")
print(f"difference    : {100*(R_inf-R_H)/R_H:.4f}%")
print(f"\npredicted H-alpha (vacuum): {1e9/(R_H*(1/4-1/9)):.3f} nm   [accepted 656.46 nm]")


## Simulated Balmer measurements

Wavenumbers for $n = 3\ldots30$ from the exact formula, plus 0.02% noise representing
wavelength calibration error in a good spectrometer.

*Simulated, stated plainly — so we know ground truth and can quote an honest error.
The physics is exact.*


In [ ]:
rng = np.random.default_rng(3)

n = np.arange(3, 31).astype(float)
wavenumber = R_H * (1 / 4 - 1 / n ** 2) * (1 + rng.normal(0, 2e-4, len(n)))

balmer = save_csv(pd.DataFrame({
    "n": n.astype(int),
    "wavenumber_per_m": wavenumber,
    "wavelength_nm": 1e9 / wavenumber,
}), "balmer_lines.csv")

print(f"{len(balmer)} lines, "
      f"{balmer.wavelength_nm.min():.1f} - {balmer.wavelength_nm.max():.1f} nm")
balmer.head()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(balmer.n, balmer.wavelength_nm, s=30)
ax[0].set_xlabel("n"); ax[0].set_ylabel("wavelength (nm)")
ax[0].set_title("Lines crowd toward the series limit")

ax[1].scatter(1 / balmer.n ** 2, balmer.wavenumber_per_m, s=30)
ax[1].set_xlabel(r"$1/n^2$"); ax[1].set_ylabel("wavenumber (1/m)")
ax[1].set_title("Rydberg coordinates — straight line")

plt.tight_layout()
plt.show()


## Stage 1 — find the form


In [ ]:
X = balmer.n.values.astype(float).reshape(-1, 1)
y = balmer.wavenumber_per_m.values

model = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=0).fit(X, y)

print("discovered     :", model.formulas())
print(f"R^2            : {model.score(X, y):.8f}")
print(f"fdr_controlled_: {model.fdr_controlled_}")


`(1/x0)^2` is $1/n^2$ — precisely the Rydberg coordinate.


## Stage 2 — measure R_H twice over

We **keep** the intercept, because physics says there is one ($R_H/4$). That gives
two independent estimates of the same constant.


In [ ]:
ridge_slope = raw_coef(model)[0]
shrink = len(n) / (len(n) + 1.0)

ols = LinearRegression().fit((1 / balmer.n.values ** 2).reshape(-1, 1), y)
R_from_slope = -ols.coef_[0]
R_from_intercept = 4 * ols.intercept_

print(f"{'source':<28}{'R_H (1/m)':>16}{'error':>11}")
print("-" * 56)
print(f"{'beamfeat ridge slope':<28}{-ridge_slope:>16,.1f}{pct_err(-ridge_slope, R_H):>10.3f}%")
print(f"{'OLS slope':<28}{R_from_slope:>16,.1f}{pct_err(R_from_slope, R_H):>10.4f}%")
print(f"{'OLS intercept x 4':<28}{R_from_intercept:>16,.1f}{pct_err(R_from_intercept, R_H):>10.4f}%")
print(f"{'true R_H':<28}{R_H:>16,.1f}")
print(f"\nridge shrink: predicted {shrink:.4f}, observed {-ridge_slope/R_H:.4f}")


Slope and intercept agree with each other and with truth to about 0.01%. **Two
independent routes to the same constant agreeing is far stronger evidence than a high
R².** Build that kind of internal check in whenever the physics offers one.


## Predicting a line that was never fitted

The real test of a recovered law is extrapolation. Our fit used $n \geq 3$; predict
the Balmer limit ($n \to \infty$, the series edge).


In [ ]:
limit_predicted = 1.0 / (R_from_slope / 4)
limit_true = 1.0 / (R_H / 4)

print(f"Balmer limit predicted : {limit_predicted*1e9:.3f} nm")
print(f"Balmer limit accepted  : {limit_true*1e9:.3f} nm  (364.7 vacuum, 364.6 in air)")
print(f"error                  : {pct_err(limit_predicted, limit_true):+.4f}%")

h_alpha = 1.0 / (R_from_slope * (1/4 - 1/9))
print(f"\nH-alpha predicted      : {h_alpha*1e9:.3f} nm   [accepted 656.46 nm]")


## Beyond Balmer: three series at once

The full Rydberg formula has *two* quantum numbers. Adding the Lyman ($n_1=1$,
ultraviolet) and Paschen ($n_1=3$, infrared) series tests whether the structure holds
across regimes a single spectrometer would never see together.


In [ ]:
rows = []
for n1, series in [(1, "Lyman"), (2, "Balmer"), (3, "Paschen")]:
    for n2 in range(n1 + 1, n1 + 15):
        wn = R_H * (1 / n1 ** 2 - 1 / n2 ** 2) * (1 + rng.normal(0, 2e-4))
        rows.append((series, n1, n2, wn, 1e9 / wn))

full = save_csv(pd.DataFrame(
    rows, columns=["series", "n1", "n2", "wavenumber_per_m", "wavelength_nm"]),
    "hydrogen_series.csv")

full.groupby("series").agg(lines=("n2", "size"),
                           lam_min=("wavelength_nm", "min"),
                           lam_max=("wavelength_nm", "max")).round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for series, grp in full.groupby("series"):
    ax.scatter(grp.wavelength_nm, [series] * len(grp), s=30, alpha=.7)
ax.set_xscale("log")
ax.set_xlabel("wavelength (nm, log scale)")
ax.set_title("Three series span UV through infrared")
plt.tight_layout()
plt.show()


In [ ]:
# Stage 2 on the two-quantum-number basis. No intercept: the formula has none.
Z = np.column_stack([1 / full.n1 ** 2, 1 / full.n2 ** 2])
ols_full = LinearRegression(fit_intercept=False).fit(Z, full.wavenumber_per_m)

print(f"coefficient on 1/n1^2 : {ols_full.coef_[0]:>14,.1f}   "
      f"(expect +R_H)  error {pct_err(ols_full.coef_[0], R_H):+.4f}%")
print(f"coefficient on 1/n2^2 : {ols_full.coef_[1]:>14,.1f}   "
      f"(expect -R_H)  error {pct_err(-ols_full.coef_[1], R_H):+.4f}%")
print(f"true R_H              : {R_H:>14,.1f}")


Both coefficients recover $R_H$ with opposite signs, to within 0.05% — across
wavelengths from 91 nm to 1.9 µm. The same constant governs all three series, which
is the physical content of the Rydberg formula.


## Takeaways

1. Given only the quantum number, the search recovered $1/n^2$ unaided.
2. Slope and intercept gave two independent estimates of $R_H$ agreeing to ~0.01%.
3. **Use the right constant.** $R_\infty$ and $R_H$ differ by 0.055%, which exceeds
   our precision — using the wrong one would have dominated the error.
4. Ridge shrinkage at $n=28$ was the predicted 0.966, removed by the OLS refit.
5. Extrapolating to the Balmer limit — never fitted — lands within 0.01%.
6. Across three series the same $R_H$ appears with opposite signs on $1/n_1^2$ and
   $1/n_2^2$, exactly as the full formula requires.
